# index-by-tensor — ex1: embedding lookup with 2d index tensor

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `index-by-tensor`. Running the final beacon cell reports progress against the `PyTorch: index by tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: index by tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`index-by-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "index-by-tensor"
DD_SUBTOPIC = "PyTorch: index by tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## advanced indexing: index a tensor BY a tensor — quick refresher

Slice indexing: `x[1:4]` produces a view, shape derived from the slice. Tensor indexing: `x[idx]` where `idx` is a LongTensor produces a **gather**, and the output shape becomes `idx.shape + x.shape[1:]`.

**Key rule.** When you index a `(N, D)` tensor with a `(K,)` index, you get `(K, D)`. When you index with a `(K, L)` index, you get `(K, L, D)`. The index tensor's shape **becomes** the leading shape of the output — you're broadcasting the gather over an arbitrary index layout.

Use it for: embedding lookups (`embed_table[token_ids]`), gathering minibatch examples by id, scattering predictions back to original-data order.

### Exercise 1 — embedding lookup with 2d index tensor

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `embed[idx]` advanced indexing where `idx` is a 2-D LongTensor of token ids, returning the corresponding `(B, T, D)` embedding lookup.
> Keywords: advanced-indexing, gather, embedding, lookup
> ```

**KCs targeted:** `index-by-long-tensor`, `advanced-indexing-shape-rule`

Implement `ex1_embedding_lookup(embed, idx)`. The canonical embedding-table lookup:

1. `embed` is a `(V, D)` embedding table — `V` is vocab size, `D` is embedding dim.
2. `idx` is a `(B, T)` LongTensor of token ids, each in `[0, V)`.
3. Use advanced indexing `embed[idx]` to produce a `(B, T, D)` tensor where `out[b, t] == embed[idx[b, t]]`.

The index tensor's shape becomes the LEADING shape of the output. Do not loop — use the single-expression `embed[idx]` form.

Input: `embed: (V, D) Tensor`, `idx: (B, T) LongTensor`.
Output: `(B, T, D) Tensor`.

In [ ]:
def ex1_embedding_lookup(embed: Tensor, idx: Tensor) -> Tensor:
    return embed[idx]


<details><summary>Solution</summary>

```python
def ex1_embedding_lookup(embed: Tensor, idx: Tensor) -> Tensor:
    return embed[idx]
```

**The whole skill is the one-liner.** Once you've internalized the shape rule — `idx.shape + embed.shape[1:]` — the syntax is trivial. The drill is teaching the rule, not the syntax.

**Why this is NOT a slice.** Slices `embed[1:4]` produce views with no copy. Advanced indexing with a tensor produces a NEW tensor — gradients flow through the gather op, and the result does not alias `embed`'s storage.

**Why this is the foundation of `nn.Embedding`.** `nn.Embedding` is essentially a learnable `(V, D)` parameter and an `embed[idx]` forward. No magic — just advanced indexing with the right grads.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()